# Havnsø – Probabilistic Pressure and Injectivity Screening

This is the second decision gate after the Havnsø static-capacity notebook. In every Monte Carlo iteration it checks:

1. Is sampled static capacity at least the project target?
2. Can each well support the selected injection rate?
3. Does the screening pressure remain at or below the pressure endpoint?

An iteration succeeds only when **all three criteria pass together**.

In [ ]:
#@title Install dependencies { display-mode: "form" }
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
#@title Import libraries { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import (
    Distribution, StorageSite, TechnicalScreeningCase,
    simulate, simulate_technical_screening,
)
plt.style.use("seaborn-v0_8-whitegrid")

## What is reported and what is derived

The pressure/injectivity calculation is a **GEUS-normalized screening surrogate**, not an Eclipse 100 reproduction. GEUS Report 2020/48 publishes a reference case and qualitative sensitivities, but no transferable pressure equation, numerical outcome for every sensitivity, or probability weights.

The surrogate exactly reaches the reference endpoint when all inputs equal the 2020 base case. It scales pressure demand with cumulative mass, total field rate, inverse permeability factor, and inverse N/G. It scales the per-well injectivity limit with permeability factor and N/G.

The updated 2023 static-capacity samples remain the current capacity evidence. The older version-0 dynamic result is used only to normalize this provisional technical gate.

## Editable project controls

The default **60 Mt** target is selected near the updated GEUS 2023/38 P50 of 62.82 Mt. It is an editable screening target, not a GEUS development plan.

In [ ]:
target_mass_mt = 60.0 #@param {type:"number"}
number_of_wells = 3 #@param {type:"integer"}
rate_mtpy_per_well = 1.0 #@param {type:"number"}
iterations = 100000 #@param {type:"integer"}
random_seed = 42 #@param {type:"integer"}

# GEUS sensitivity values converted into a provisional PERT model.
permeability_factor = (0.5, 1.0, 2.0)

In [ ]:
#@title Show GEUS reference values and evidence status { display-mode: "form" }
evidence_table = pd.DataFrame([
    ["Initial datum pressure", 130, "bar", "GEUS reported", "2020/48 Table 1"],
    ["Initial temperature", 42, "°C", "GEUS reported", "2020/48 Table 1"],
    ["Reference wells", 3, "count", "GEUS reported", "2020/48 base case"],
    ["Reference rate per well", 1, "Mt/year/well", "GEUS reported", "2020/48 base case"],
    ["Reference duration", 90, "years", "GEUS reported", "2020/48 base case"],
    ["Reference injected mass", 270, "Mt", "GEUS reported", "2020/48 base case"],
    ["Pressure endpoint", 240, "bar absolute", "Interpretation of GEUS wording", "See note below"],
    ["Reference N/G", 0.5, "fraction", "GEUS reported", "2020/48 base model"],
    ["N/G sensitivity", 0.9, "fraction", "GEUS reported", "2020/48 sensitivity"],
    ["Permeability factors", "0.5 / 1 / 2", "multiplier", "GEUS scenarios; probability model derived", "2020/48 sensitivity"],
    ["Project target", target_mass_mt, "Mt", "User-editable assumption", "Default near 2023/38 P50"],
], columns=["Parameter", "Value", "Unit", "Status", "Source / basis"])
evidence_table

**Pressure interpretation:** the English summary calls 240 bar an “overpressure”. This notebook treats 240 bar as the absolute endpoint pressure. Adding 240 bar to the 130-bar initial pressure would conflict with the report's statement that the 75%-of-lithostatic fracture constraint was respected. Keep this interpretation visible until the original simulation files or an updated model resolve the terminology.

In [ ]:
#@title Run static capacity Monte Carlo { display-mode: "form" }
site = StorageSite(
    name="Havnsø – Gassum Formation – Scenario 1",
    grv=Distribution.pert(2.9, 5.0, 8.0),
    net_to_gross=Distribution.pert(0.60, 0.75, 0.90),
    porosity=Distribution.pert(0.175, 0.219, 0.263),
    co2_density=Distribution.pert(663.86, 698.8, 768.68),
    storage_efficiency=Distribution.pert(0.05, 0.10, 0.20),
)
capacity_result = simulate(site, iterations=iterations, seed=random_seed)

In [ ]:
#@title Run pressure and injectivity screening { display-mode: "form" }
technical_case = TechnicalScreeningCase(
    name=f"Havnsø – {target_mass_mt:g} Mt technical screening",
    target_mass_mt=target_mass_mt,
    wells=number_of_wells,
    rate_mtpy_per_well=rate_mtpy_per_well,
    permeability_factor=Distribution.pert(*permeability_factor),
    initial_pressure_bar=130.0,
    pressure_limit_bar=240.0,
    reference_mass_mt=270.0,
    reference_wells=3,
    reference_rate_mtpy_per_well=1.0,
    reference_net_to_gross=0.5,
)
technical_result = simulate_technical_screening(
    technical_case, capacity_result, seed=random_seed + 1
)

In [ ]:
#@title Show integrated technical-screening result { display-mode: "form" }
summary = technical_result.summary()
result_table = pd.DataFrame([
    ["Project target", f"{target_mass_mt:.1f} Mt"],
    ["Injection duration", f"{technical_result.duration_years:.1f} years"],
    ["Capacity passes", f"{summary['capacity_pass_probability']:.1%}"],
    ["Injectivity passes", f"{summary['injectivity_pass_probability']:.1%}"],
    ["Pressure passes", f"{summary['pressure_pass_probability']:.1%}"],
    ["All criteria pass (technical success)", f"{summary['success_probability']:.1%}"],
    ["At least one criterion fails", f"{summary['failure_probability']:.1%}"],
    ["P90 final pressure", f"{summary['p90_final_pressure_bar']:.1f} bar"],
    ["P50 final pressure", f"{summary['p50_final_pressure_bar']:.1f} bar"],
], columns=["Metric", "Result"])
result_table

In [ ]:
#@title Show criterion pass probabilities { display-mode: "form" }
fig, ax = technical_result.plot_criteria()
plt.show()

In [ ]:
#@title Show final-pressure distribution and limit { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(technical_result.final_pressure_bar, bins=50, color="#9ecae1", edgecolor="white")
ax.axvline(240, color="#c00000", linestyle="--", linewidth=2, label="Pressure endpoint: 240 bar")
ax.set(xlabel="Screening final pressure (bar absolute)", ylabel="Iterations", title=technical_case.name)
ax.legend()
plt.show()

## Interpretation and next calibration step

The combined success percentage is a **probabilistic screening result**, not yet a calibrated geological-risk probability. Capacity uncertainty comes from GEUS 2023/38. The permeability-factor probability distribution is derived from the 0.5× and 2× sensitivity scenarios in GEUS 2020/48; GEUS did not assign probabilities to them.

Before using the result for a project decision, replace this surrogate with an updated dynamic reservoir model based on the 2023 geometry and obtain numerical pressure responses, site-specific permeability/relative-permeability data, fracture-pressure measurements, and well-design constraints.